## Disneyland Review Rating Prediction

Given *reviews of Disneyland*, let's try to predict the **rating** associated with a given review.

We will use a TensorFlow/Keras text model with word embeddings to make our predictions.

Data source: https://www.kaggle.com/datasets/arushchillar/disneyland-reviews

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import tensorflow as tf

In [3]:
data = pd.read_csv('archive/DisneylandReviews.csv', encoding='latin-1')

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42656 entries, 0 to 42655
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Review_ID          42656 non-null  int64 
 1   Rating             42656 non-null  int64 
 2   Year_Month         42656 non-null  object
 3   Reviewer_Location  42656 non-null  object
 4   Review_Text        42656 non-null  object
 5   Branch             42656 non-null  object
dtypes: int64(2), object(4)
memory usage: 2.0+ MB


### Preprocessing

In [5]:
df = data.copy()

In [6]:
df

,Review_ID,Rating,Year_Month,Reviewer_Location,Review_Text,Branch
0,670772142,4,2019-4,Australia,If you've ever been to Disneyland anywhere you...,Disneyland_HongKong
1,670682799,4,2019-5,Philippines,Its been a while since d last time we visit HK...,Disneyland_HongKong
2,670623270,4,2019-4,United Arab Emirates,Thanks God it wasn t too hot or too humid wh...,Disneyland_HongKong
3,670607911,4,2019-4,Australia,HK Disneyland is a great compact park. Unfortu...,Disneyland_HongKong
4,670607296,4,2019-4,United Kingdom,"the location is not in the city, took around 1...",Disneyland_HongKong
...,...,...,...,...,...,...
42651,1765031,5,missing,United Kingdom,i went to disneyland paris in july 03 and thou...,Disneyland_Paris
42652,1659553,5,missing,Canada,2 adults and 1 child of 11 visited Disneyland ...,Disneyland_Paris
42653,1645894,5,missing,South Africa,My eleven year old daughter and myself went to...,Disneyland_Paris
42654,1618637,4,missing,United States,"This hotel, part of the Disneyland Paris compl...",Disneyland_Paris


In [9]:
# Limit data to only the review and rating column
y = df.loc[:, 'Rating']

In [10]:
X = df['Review_Text']

In [11]:
X

0        If you've ever been to Disneyland anywhere you...
1        Its been a while since d last time we visit HK...
2        Thanks God it wasn   t too hot or too humid wh...
3        HK Disneyland is a great compact park. Unfortu...
4        the location is not in the city, took around 1...
                               ...                        
42651    i went to disneyland paris in july 03 and thou...
42652    2 adults and 1 child of 11 visited Disneyland ...
42653    My eleven year old daughter and myself went to...
42654    This hotel, part of the Disneyland Paris compl...
42655    I went to the Disneyparis resort, in 1996, wit...
Name: Review_Text, Length: 42656, dtype: object

In [12]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

In [13]:
X_train

20780    I love any and all things Disney! I went this ...
791      Easy to get to using the MTR, great for little...
19394    We visited both the California Adventure park ...
32755    This awesome place is not only fun filled for ...
38577    Just got back from disneyland paris and wasnt ...
                               ...                        
7813     Ocean Park far more value for money.  Disneyla...
32511    I went with a friend to stay for 4 nights so w...
5192     Disneyland in Hong Kong is a beautiful park wi...
12172    I have various season passes for California th...
33003    It's Disney, it's magical. we spent 3 nights s...
Name: Review_Text, Length: 29859, dtype: object

In [14]:
# Fit tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

tokenizer

In [15]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'a': 4,
 'of': 5,
 'we': 6,
 'in': 7,
 'for': 8,
 'was': 9,
 'it': 10,
 'is': 11,
 'i': 12,
 'you': 13,
 'park': 14,
 'but': 15,
 'at': 16,
 'were': 17,
 'on': 18,
 'that': 19,
 'are': 20,
 'as': 21,
 'with': 22,
 'disney': 23,
 'rides': 24,
 'not': 25,
 'disneyland': 26,
 'there': 27,
 'have': 28,
 'so': 29,
 'this': 30,
 'all': 31,
 'time': 32,
 'day': 33,
 'be': 34,
 'my': 35,
 'had': 36,
 'they': 37,
 'get': 38,
 'if': 39,
 'our': 40,
 'go': 41,
 'very': 42,
 'one': 43,
 'ride': 44,
 'can': 45,
 'just': 46,
 'great': 47,
 'your': 48,
 'from': 49,
 'or': 50,
 'do': 51,
 'food': 52,
 'more': 53,
 'which': 54,
 'would': 55,
 'kids': 56,
 'when': 57,
 'place': 58,
 'good': 59,
 'will': 60,
 'some': 61,
 'only': 62,
 'an': 63,
 'really': 64,
 'out': 65,
 'like': 66,
 'visit': 67,
 "it's": 68,
 'see': 69,
 'up': 70,
 'no': 71,
 'went': 72,
 'much': 73,
 'people': 74,
 'been': 75,
 'about': 76,
 'also': 77,
 'long': 78,
 'than': 79,
 'back': 80,
 '2': 81,


In [17]:
print("Vocab Length:", len(tokenizer.word_index) + 1)

Vocab Length: 37846


In [18]:
def get_sequences(texts, tokenizer, train=True, max_seq_length=None):
    sequences = tokenizer.texts_to_sequences(texts)

    if train == True:
        max_seq_length = np.max(list(map(len, sequences)))

    sequences = pad_sequences(sequences, maxlen=max_seq_length, padding='post')
    
    return sequences

In [19]:
# Convert texts to sequences
X_train = get_sequences(X_train, tokenizer, train=True)
X_test = get_sequences(X_test, tokenizer, train=False, max_seq_length=X_train.shape[1])

In [20]:
X_train

array([[ 12, 154, 159, ...,   0,   0,   0],
       [330,   3,  38, ...,   0,   0,   0],
       [  6, 168, 193, ...,   0,   0,   0],
       ...,
       [ 26,   7, 251, ...,   0,   0,   0],
       [ 12,  28, 989, ...,   0,   0,   0],
       [ 68,  23,  68, ...,   0,   0,   0]],
      shape=(29859, 3958), dtype=int32)

In [21]:
X_test

array([[   6,  168,    8, ...,    0,    0,    0],
       [   6,  443,   18, ...,    0,    0,    0],
       [ 381,   64,  206, ...,    0,    0,    0],
       ...,
       [1416,    1,  653, ...,    0,    0,    0],
       [   6, 2503,   49, ...,    0,    0,    0],
       [   6,  168,   22, ...,    0,    0,    0]],
      shape=(12797, 3958), dtype=int32)

In [22]:
y_train

20780    5
791      3
19394    5
32755    5
38577    3
        ..
7813     3
32511    5
5192     5
12172    5
33003    5
Name: Rating, Length: 29859, dtype: int64

In [23]:
y_test

12008    5
42394    1
24748    5
42609    1
10719    5
        ..
37120    4
33226    4
3764     5
22423    5
293      4
Name: Rating, Length: 12797, dtype: int64

### Training

In [24]:
X_train.shape

(29859, 3958)

In [30]:
inputs = tf.keras.Input(shape=(3958,))
x = tf.keras.layers.Embedding(
    input_dim = 37846,
    output_dim = 64
)(inputs)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='linear')(x)

2026-07-17 21:44:33.354758: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 129695744 exceeds 10% of free system memory.
2026-07-17 21:44:33.524439: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 129695744 exceeds 10% of free system memory.
2026-07-17 21:44:33.598601: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 129695744 exceeds 10% of free system memory.


In [31]:
model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='mse'
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=100,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
    ]
)

Epoch 1/100


2026-07-17 21:47:24.251238: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 378178984 exceeds 10% of free system memory.
2026-07-17 21:47:25.100972: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 129695744 exceeds 10% of free system memory.


747/747 ━━━━━━━━━━━━━━━━━━━━ 420s 561ms/step - loss: 91.5605 - val_loss: 4.9258
Epoch 2/100
747/747 ━━━━━━━━━━━━━━━━━━━━ 431s 577ms/step - loss: 1.1308 - val_loss: 2.5580
Epoch 3/100
747/747 ━━━━━━━━━━━━━━━━━━━━ 430s 576ms/step - loss: 0.6475 - val_loss: 0.6878
Epoch 4/100
747/747 ━━━━━━━━━━━━━━━━━━━━ 430s 576ms/step - loss: 0.4067 - val_loss: 0.6771
Epoch 5/100
747/747 ━━━━━━━━━━━━━━━━━━━━ 449s 601ms/step - loss: 0.2391 - val_loss: 0.7009
Epoch 6/100
747/747 ━━━━━━━━━━━━━━━━━━━━ 494s 590ms/step - loss: 0.1468 - val_loss: 0.7290
Epoch 7/100
747/747 ━━━━━━━━━━━━━━━━━━━━ 447s 597ms/step - loss: 0.0934 - val_loss: 0.7302


### Results

In [32]:
y_pred = np.squeeze(model.predict(X_test))

rmse = np.sqrt(np.mean((y_test - y_pred)**2))

print("RMSE: {:.2f}".format(rmse))

400/400 ━━━━━━━━━━━━━━━━━━━━ 18s 46ms/step
RMSE: 0.81


In [38]:
r2 = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2))
r2

np.float64(0.41445094168844254)

In [39]:
print("R^2 Score: {:.5f}".format(r2))

R^2 Score: 0.41445
